# A2C Wordle Solver - Model Evaluation

This notebook evaluates a trained A2C Wordle model with comprehensive metrics and visualizations.

## What This Notebook Does

1. **Loads the trained model** from a checkpoint
2. **Evaluates on multiple word sets:**
   - Training vocabulary (check if agent learned)
   - Held-out words (check generalization)
   - Full vocabulary (if applicable)
3. **Generates detailed metrics:**
   - Win rate
   - Average turns to win
   - Turn distribution (histogram)
   - Per-word results
4. **Creates visualizations** and saves results to CSV/JSON

## Setup Instructions

1. Upload the checkpoint file to Colab or mount Google Drive
2. Install dependencies (if needed)
3. Run the evaluation cells below

## 1. Setup & Installation

In [ ]:
# May need to clone the repo first

In [ ]:
# Install dependencies (if not already installed)
!pip install -q pytorch-lightning gymnasium numpy pandas matplotlib tqdm fire

In [ ]:
# Set up paths
import sys
sys.path.insert(0, "/content/wordle-solver-a2c")
sys.path.insert(0, "/content/wordle-solver-a2c/deep_rl")

# Verify imports work
import torch
import gymnasium as gym
from pathlib import Path

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Upload or Locate the Checkpoint

In [ ]:
# Option A: Upload checkpoint file directly
from google.colab import files

print("Upload the .ckpt checkpoint file:")
uploaded = files.upload()

# Get the uploaded filename
checkpoint_path = list(uploaded.keys())[0]
print(f"\n✓ Loaded checkpoint: {checkpoint_path}")

In [ ]:
# Option B: Use checkpoint from Google Drive
# Uncomment to mount Google Drive:

# from google.colab import drive
# drive.mount('/content/drive')

# checkpoint_path = '/content/drive/MyDrive/wordle_checkpoints/a2c-epoch=10.ckpt'
# print(f"Using checkpoint: {checkpoint_path}")

In [ ]:
# Option C: Use checkpoint from cloned repo (if you have checkpoints in git)
# checkpoint_path = 'checkpoints/a2c-epoch=10.ckpt'
# print(f"Using checkpoint: {checkpoint_path}")

## 3. Quick Evaluation (Command Line)

In [ ]:
# Run full evaluation script
!python evaluate_model.py \
    --checkpoint {checkpoint_path} \
    --output_dir evaluation_results

## 4. View Results

In [ ]:
# View the summary JSON
import json

with open('evaluation_results/summary.json', 'r') as f:
    summary = json.load(f)

print(json.dumps(summary, indent=2))

In [ ]:
# Load and view detailed results
import pandas as pd

df_train = pd.read_csv('evaluation_results/training_words_detailed.csv')
print(f"Training set results: {len(df_train)} words")
print(f"\nWin rate: {df_train['win'].mean()*100:.2f}%")
print(f"Avg turns: {df_train['turns'].mean():.2f}")
print(f"\nFirst 10 results:")
df_train.head(10)

In [ ]:
# Show failed words
failed = df_train[df_train['win'] == False]
print(f"Failed words: {len(failed)}")
if len(failed) > 0:
    print("\nFailed attempts:")
    print(failed[['goal_word', 'guesses_str']])

In [ ]:
# Display generated plots
from IPython.display import Image, display

print("Turn Distribution - Training Words:")
display(Image('evaluation_results/training_turn_distribution.png'))

print("\nComparison Across Evaluation Sets:")
display(Image('evaluation_results/comparison.png'))

## 5. Interactive Evaluation (Programmatic)

In [ ]:
# Load model for interactive use
import a2c.play

model, agent, env = a2c.play.load_from_checkpoint(checkpoint_path)
print(f"✓ Loaded model")
print(f"  Vocabulary size: {len(env.words)}")
print(f"  Allowable goals: {env.allowable_words}")

In [ ]:
# Test on specific words
test_words = ['CRANE', 'SLATE', 'AUDIO', 'PIZZA', 'FUZZY']  # Modify as needed

for word in test_words:
    try:
        win, outcomes = a2c.play.goal(agent, env, word)
        guesses = ' → '.join([g for g, _ in outcomes])
        result = "✓ WIN" if win else "✗ LOSS"
        print(f"{word}: {result} in {len(outcomes)} turns | {guesses}")
    except ValueError:
        print(f"{word}: Not in vocabulary")

In [ ]:
# Analyze opening move preferences
from collections import Counter

opening_moves = []
for word in env.words[:100]:  # Sample 100 games
    try:
        win, outcomes = a2c.play.goal(agent, env, word)
        if outcomes:
            opening_moves.append(outcomes[0][0])
    except:
        pass

move_counts = Counter(opening_moves)
print("\nTop 10 Opening Moves:")
for move, count in move_counts.most_common(10):
    print(f"  {move}: {count} times ({count/len(opening_moves)*100:.1f}%)")

## 6. Custom Analysis

In [ ]:
# Analyze performance by word characteristics
import matplotlib.pyplot as plt

df = df_train.copy()

# Add features
df['has_duplicate_letters'] = df['goal_word'].apply(lambda w: len(set(w)) < len(w))
df['num_vowels'] = df['goal_word'].apply(lambda w: sum(1 for c in w if c in 'AEIOU'))

# Compare win rates
print("Win rate by word characteristics:")
print(f"  Words with duplicate letters: {df[df['has_duplicate_letters']]['win'].mean()*100:.1f}%")
print(f"  Words without duplicates: {df[~df['has_duplicate_letters']]['win'].mean()*100:.1f}%")
print()
print("Win rate by vowel count:")
for n in sorted(df['num_vowels'].unique()):
    subset = df[df['num_vowels'] == n]
    if len(subset) > 0:
        print(f"  {n} vowels: {subset['win'].mean()*100:.1f}% (n={len(subset)})")

In [ ]:
# Plot average turns by vowel count
vowel_stats = df.groupby('num_vowels').agg({
    'turns': 'mean',
    'win': 'mean'
}).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(vowel_stats['num_vowels'], vowel_stats['win']*100, color='green', alpha=0.7)
axes[0].set_xlabel('Number of Vowels')
axes[0].set_ylabel('Win Rate (%)')
axes[0].set_title('Win Rate by Vowel Count')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(vowel_stats['num_vowels'], vowel_stats['turns'], color='steelblue', alpha=0.7)
axes[1].set_xlabel('Number of Vowels')
axes[1].set_ylabel('Average Turns')
axes[1].set_title('Average Turns by Vowel Count')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Download Results

In [ ]:
# Zip all results for download
!zip -r evaluation_results.zip evaluation_results/

# Download the zip file
from google.colab import files
files.download('evaluation_results.zip')

print("✓ Results packaged and downloading...")

## Summary

This notebook evaluated the A2C Wordle model with:
- **Win rate**: Percentage of games won
- **Average turns**: How many guesses needed (all games and wins only)
- **Turn distribution**: Histogram showing spread of performance
- **Per-word analysis**: Detailed results for every word tested
- **Comparison plots**: Visual comparison across different word sets

### Key Metrics to Report:
1. **Training set win rate** (shows if model learned)
2. **Held-out win rate** (shows generalization)
3. **Average turns to win** (quality of strategy)
4. **Turn distribution** (consistency of performance)

### Interpreting Results:
- **Good model**: 85-95% win rate, 4-5 avg turns
- **Expert-level**: 95%+ win rate, 3.5-4 avg turns
- **Generalization**: Training win rate should be within 5-10% of held-out win rate

All results are saved in `evaluation_results/` directory!